# Datum sensitivity and a defensible interpretation

**Time:** 40 minutes · **Prerequisites:** chapters 03–04 or basic Python tables. No private input or GIS installation is required.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/17_datum_sensitivity.ipynb)

**Reading the book?** It shows saved results. You can zoom and rotate Plotly figures here; Python-backed selectors, sliders and uploads require running the notebook in JupyterLite.

Wait for **Python (Pyodide) | Idle**, then choose **Run → Run All Cells**. Public results below reflect the bundled aggregate artifact; unavailable models are labeled. Fictional exercises remain explicitly separate from the study.

In [1]:
import sys
from pathlib import Path
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'plotly', 'ipywidgets'])
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from coastlab import show, land_traces
from naturelab import (msl_elevation, screening_summary, paired_comparison,
                       load_public, load_sources, coverage_frame, fictional_samples,
                       aggregate_explorer, sensitivity_controls, terrain_explorer)
public = load_public()
sources = load_sources()
print('Ready: Nature 2026 — public aggregates and explicitly fictional exercises.')
print('Current sampling status:', public['status'])

Ready: Nature 2026 — public aggregates and explicitly fictional exercises.
Current sampling status: partial


## Objectives: test sensitivity without inventing a forecast

Explore how a hypothetical height offset changes a threshold count. Separate this controlled experiment from a new DEM comparison. Use the local sample interface only if authorized, keeping private outputs out of publication.

The following 3D surfaces and ten example rows are **entirely fictional**. Their horizontal axes are arbitrary teaching units and their vertical separation is exaggerated. They are not geoid grids, facility observations, real coastlines or evidence about a particular city.

In [2]:
x = np.linspace(-2, 2, 32)
xx, yy = np.meshgrid(x, x)
geoid = .15*np.sin(xx)*np.cos(yy)
sea = geoid + .7 + .1*xx
ground = 1.3 + .2*xx + .25*np.cos(yy)
fig = go.Figure()
for name, surface, color, opacity in [
    ('Fictional geoid', geoid, '#825aa6', .6),
    ('Fictional mean sea surface', sea, '#279fb5', .65),
    ('Fictional terrain', ground, '#b99367', .85)]:
    fig.add_surface(x=xx, y=yy, z=surface, name=name, showscale=False,
                    colorscale=[[0, color], [1, color]], opacity=opacity,
                    hovertemplate=name + '<br>height %{z:.2f} m<extra></extra>')
show(fig.update_layout(title='FICTIONAL reference surfaces — not a geographic map',
                       scene=dict(xaxis_title='Arbitrary x', yaxis_title='Arbitrary y',
                                  zaxis_title='Height above fictional ellipsoid (m)')))
display(pd.DataFrame({'surface': ['geoid', 'mean sea', 'terrain'],
                      'height_at_x0_y0_m': [0, .7, 1.55]}))

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="50b152eb-1039-4f66-98d5-6d879b31b3c5" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("50b152eb-1039-4f66-98d5-6d879b31b3c5")) { Plotly.newPlot( "50b152eb-1039-4f66-98d5-6d879b31b3c5", [{"colorscale":[[0,"#825aa6"],[1,"#825aa6"]],"hovertemplate":"Fictional geoid\u003cbr\u003eheight %{z:.2f} m\u003cextra\u003e\u003c\u002fextra\u003e","name":"Fictional geoid","opacity":0.6,"showscale":false,"x":{"dtype":"f8","bdata":"AAAAAAAAAMDwvffee+\u002f9v99777333vu\u002fzjnnnHPO+b++99577733v6611lprrfW\u002fnXPOOeec87+MMcYYY4zxv\u002fjee++99+6\u002f2FprrbXW6r+21lprrbXmv5RSSimllOK\u002f6Jxzzjnn3L+olFJKKaXUv8gYY4wxxsi\u002fgBBCCCGEsL+AEEIIIYSwP8AYY4wxxsg\u002foJRSSiml1D\u002fonHPOOefcP5RSSimllOI\u002ftNZaa6215j\u002fYWmuttdbqP\u002fjee++99+4\u002fjDHGGGOM8T+cc84555zzP6y11lprrfU\u002fvvfee++99z\u002fOOeecc875P95777333vs\u002f8L333nvv\u002fT8AAAAAAAAAQAAAAAAAAADA8L333nvv\u002fb\u002ffe++99977v84555xzzvm\u002fvvfee++997+utdZaa631v51zzjnnnPO\u002fjDHGGGOM8b\u002f43nvvvffuv9haa6211uq\u002fttZaa6215r+UUkoppZTiv+icc84559y\u002fqJRSSiml1L\u002fIGGOMMcbIv4AQQgghhLC\u002fgBBCCCGEsD\u002fAGGOMMcbIP6CUUkoppdQ\u002f6Jxzzjnn3D+UUkoppZTiP7TWWmutteY\u002f2FprrbXW6j\u002f43nvvvffuP4wxxhhjjPE\u002fnHPOOeec8z+stdZaa631P7733nvvvfc\u002fzjnnnHPO+T\u002fee++99977P\u002fC999577\u002f0\u002fAAAAAAAAAEAAAAAAAAAAwPC999577\u002f2\u002f33vvvffe+7\u002fOOeecc875v7733nvvvfe\u002frrXWWmut9b+dc84555zzv4wxxhhjjPG\u002f+N5777337r\u002fYWmuttdbqv7bWWmuttea\u002flFJKKaWU4r\u002fonHPOOefcv6iUUkoppdS\u002fyBhjjDHGyL+AEEIIIYSwv4AQQgghhLA\u002fwBhjjDHGyD+glFJKKaXUP+icc84559w\u002flFJKKaWU4j+01lprrbXmP9haa6211uo\u002f+N5777337j+MMcYYY4zxP5xzzjnnnPM\u002frLXWWmut9T++99577733P84555xzzvk\u002f3nvvvffe+z\u002fwvffee+\u002f9PwAAAAAAAABAAAAAAAAAAMDwvffee+\u002f9v99777333vu\u002fzjnnnHPO+b++99577733v6611lprrfW\u002fnXPOOeec87+MMcYYY4zxv\u002fjee++99+6\u002f2FprrbXW6r+21lprrbXmv5RSSimllOK\u002f6Jxzzjnn3L+olFJKKaXUv8gYY4wxxsi\u002fgBBCCCGEsL+AEEIIIYSwP8AYY4wxxsg\u002foJRSSiml1D\u002fonHPOOefcP5RSSimllOI\u002ftNZaa6215j\u002fYWmuttdbqP\u002fjee++99+4\u002fjDHGGGOM8T+cc84555zzP6y11lprrfU\u002fvvfee++99z\u002fOOeecc875P95777333vs\u002f8L333nvv\u002fT8AAAAAAAAAQAAAAAAAAADA8L333nvv\u002fb\u002ffe++99977v84555xzzvm\u002fvvfee++997+utdZaa631v51zzjnnnPO\u002fjDHGGGOM8b\u002f43nvvvffuv9haa6211uq\u002fttZaa6215r+UUkoppZTiv+icc84559y\u002fqJRSSiml1L\u002fIGGOMMcbIv4AQQgghhLC\u002fgBBCCCGEsD\u002fAGGOMMcbIP6CUUkoppdQ\u002f6Jxzzjnn3D+UUkoppZTiP7TWWmutteY\u002f2FprrbXW6j\u002f43nvvvffuP4wxxhhjjPE\u002fnHPOOeec8z+stdZaa631P7733nvvvfc\u002fzjnnnHPO+T\u002fee++99977P\u002fC999577\u002f0\u002fAAAAAAAAAEAAAAAAAAAAwPC999577\u002f2\u002f33vvvffe+7\u002fOOeecc875v7733nvvvfe\u002frrXWWmut9b+dc84555zzv4wxxhhjjPG\u002f+N5777337r\u002fYWmuttdbqv7bWWmuttea\u002flFJKKaWU4r\u002fonHPOOefcv6iUUkoppdS\u002fyBhjjDHGyL+AEEIIIYSwv4AQQgghhLA\u002fwBhjjDHGyD+glFJKKaXUP+icc84559w\u002flFJKKaWU4j+01lprrbXmP9haa6211uo\u002f+N5777337j+MMcYYY4zxP5xzzjnnnPM\u002frLXWWmut9T++99577733P84555xzzvk\u002f3nvvvffe+z\u002fwvffee+\u002f9PwAAAAAAAABAAAAAAAAAAMDwvffee+\u002f9v99777333vu\u002fzjnnnHPO+b++99577733v6611lprrfW\u002fnXPOOeec87+MMcYYY4zxv\u002fjee++99+6\u002f2FprrbXW6r+21lprrbXmv5RSSimllOK\u002f6Jxzzjnn3L+olFJKKaXUv8gYY4wxxsi\u002fgBBCCCGEsL+AEEIIIYSwP8AYY4wxxsg\u002foJRSSiml1D\u002fonHPOOefcP5RSSimllOI\u002ftNZaa6215j\u002fYWmuttdbqP\u002fjee++99+4\u002fjDHGGGOM8T+cc84555zzP6y11lprrfU\u002fvvfee++99z\u002fOOeecc875P95777333vs\u002f8L333nvv\u002fT8AAAAAAAAAQAAAAAAAAADA8L333nvv\u002fb\u002ffe++99977v84555xzzvm\u002fvvfee++997+utdZaa631v51zzjnnnPO\u002fjDHGGGOM

,surface,height_at_x0_y0_m
0,geoid,0.00
1,mean sea,0.70
2,terrain,1.55


## Move the threshold and the hypothetical height shift

Increasing the threshold includes more valid rows. Increasing the height shift raises terrain relative to a fixed threshold and generally includes fewer. The curve holds the input cohort and missingness fixed. It does not simulate tides, storms, sea-level trajectories, land subsidence, defenses or changing services.

Input values ≤−7 m and missing cells remain unavailable for every shift. A slider cannot rescue a bad cell. This is a sensitivity experiment on valid values, not a reclassification of source quality.

In [3]:
sensitivity = sensitivity_controls(allow_upload=True)
demo = fictional_samples()
base = screening_summary(demo.msl_m, 1, 0)
raised = screening_summary(demo.msl_m, 1, 1)
assert base['at_or_below'] == 4 and raised['at_or_below'] == 2
assert base['available'] == raised['available'] == 8
print('FICTIONAL worked check: 4 -> 2 when valid heights rise by 1 m; coverage stays 8/10.')

FICTIONAL worked check: 4 -> 2 when valid heights rise by 1 m; coverage stays 8/10.


## Optional private CSV, processed in your browser

The sampler creates `private-data/nature/samples.csv`. Its required interface is `uid,model,msl_m`; `original_mhhw_m` is optional. Each model has one row per supported facility. Missing heights remain blank, and excluded raw heights retain the ≤−7 rule. The upload reader ignores other columns and never displays identifiers or points. Large workbooks should be converted locally with `scripts/analyze_nature.py` first; rasterio and Excel parsing are not browser dependencies.

In **JupyterLite**, the explicit upload is read in browser memory. The helper sends no data to an external service and clears the upload widget's raw bytes after processing, including rejected files. Only the parsed dataframe remains in kernel memory for analysis. Saving or downloading an executed notebook can preserve its outputs; keep private-session notebooks local and do not publish them. **Restore fictional demo** clears the upload and replaces the private dataframe with the default fictional example. Reloading JupyterLite ends its browser kernel session. In **local/server Jupyter**, restart or shut down the kernel to clear that kernel's memory; reloading its browser page does not end the server kernel.

## Lab: what did the slider prove?

1. Keep the threshold at 1 m and move the shift from 0 to +1 m. Record the count and coverage.
2. Restore shift 0 and change the threshold to 10 m. Explain why equality is included.
3. Write two sentences distinguishing this experiment from comparing CoastalDEM-derived MHHW heights with corrected GLL MSL heights.

**Worked checks:** The default fixture changes from 4 to 2 at/below 1 m, with coverage fixed at 8/10. At ≤10 m with zero shift, 7 valid rows qualify. Equality follows the declared screening rule. The experiment changes a single numeric input while holding rows fixed; switching DEMs also changes resolution, terrain estimation, acquisition epoch and potentially coverage.

**Deliverable:** A short methods note and a chart with a fictional/private label, threshold units, coverage, source versions and uncertainty. Explain why neither a low elevation nor a threshold crossing establishes inundation, a return period, lost services or patient impacts. A useful next study would combine validated terrain, consistent water levels and a documented hydrodynamic or connectivity model with separate service-network evidence.

## Sources and evidence boundaries

Seeger & Minderhoud (2026), [*Sea level much higher than assumed in most coastal hazard assessments*](https://doi.org/10.1038/s41586-026-10196-1), with the [25 August 2026 addendum](https://doi.org/10.1038/s41586-026-11017-1). The pinned [October 2026 catalog, record 23341921](https://doi.org/10.5281/zenodo.23341921) describes corrected products and their licenses. Exact versions, URLs, roles and inspected metadata hashes are in `data/nature-sources.json`; current results are in `data/nature-aggregates.json`.

Original calculations/code here are independent MIT implementations. Corrected-raster derivatives are **CC BY-NC-SA 4.0**, distinct from the **CC BY 4.0** original lesson prose and paper. Cite the corrected product, underlying DEM and MDT/geoid sources. No private workbook, facility identifiers, names, coordinates or clinical counts are bundled. [Full source audit](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/source-audit.html) · [Data dictionary](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/data-dictionary.html) · [References](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/references.html).

In [4]:
print('Nature lesson complete: 17_datum_sensitivity')

Nature lesson complete: 17_datum_sensitivity
